In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm.auto import tqdm
from transformers import CLIPTokenizer
from diffusers import (
    StableDiffusionPipeline,
    AutoencoderKL, 
    DDPMScheduler, 
    UNet2DConditionModel
)
from diffusers.optimization import get_scheduler

from data.caption_loader import get_caption_loader
from custom.clip import CLIPTextModel

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

model_id = os.environ.get("SD_MODEL_DIR", "./models/stable-diffusion-v1-5")
dataset_id = os.environ.get("DATASET_DIR", "./data/imagenet_clip_1token")
# plugin_path = "./outputs/kv.pt"
plugin_path = None
weight_dtype = torch.float32
text_encoder = CLIPTextModel.from_pretrained(
    model_id, 
    subfolder="text_encoder", 
    dtype=weight_dtype,
    device_map="auto",
)
vae = AutoencoderKL.from_pretrained(
    model_id, 
    subfolder="vae", 
    torch_dtype=weight_dtype,
    device_map="auto",
)
unet = UNet2DConditionModel.from_pretrained(
    model_id, 
    subfolder="unet", 
    torch_dtype=weight_dtype,
    device_map="auto",
)
device = unet.device
noise_scheduler = DDPMScheduler.from_pretrained(model_id, subfolder="scheduler")
tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")

def run_pipeline(
    propmt,
    vae, 
    text_encoder, 
    tokenizer, 
    unet, 
    weight_dtype
):

    pipeline = StableDiffusionPipeline.from_pretrained(
        pretrained_model_name_or_path=model_id,
        vae = vae,
        text_encoder = text_encoder,
        tokenizer = tokenizer,
        unet = unet,
        safety_checker=None,
        torch_dtype=weight_dtype,
    )
    
    pipeline = pipeline.to(device)
    pipeline.set_progress_bar_config(disable=True)


    generator = torch.Generator(device=device).manual_seed(seed)
    image = pipeline(propmt, num_inference_steps=50, generator=generator).images[0]

    del pipeline
    torch.cuda.empty_cache()

    return image

In [ ]:
learning_rate = 2e-2
weight_decay = 0
lr_warmup_steps = 0
resolution = 512
train_batch_size = 512
num_train_epochs = 30000
lr_scheduler = "constant"

vae.requires_grad_(False)
text_encoder.requires_grad_(False)
unet.requires_grad_(False)

# activate plug-ins
text_encoder.init_plugin(
    use_act = False,
    weight_dtype = weight_dtype,
    device = device,
)

train_dataloader, dataset = get_caption_loader(
    dataset_id = dataset_id,
    tokenizer = tokenizer,
    resolution = resolution,
    train_batch_size = train_batch_size,
)

optimizer = torch.optim.Adam(
    text_encoder.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)

lr_scheduler = get_scheduler(
    lr_scheduler,
    optimizer=optimizer,
    num_warmup_steps=lr_warmup_steps,
)

progress_bar = tqdm(
    range(0, num_train_epochs),
    initial = 0,
    desc = "Steps",
)

text_encoder.eval()
module = text_encoder.plugin_model

with torch.no_grad():
    for step, batch in enumerate(train_dataloader):
        # Get the text embedding for conditioning
        for key, value in batch.items():
            batch[key] = value.to(device)
        pe = text_encoder(**batch, return_dict=False)[0][:,1]
        X = module.cache['x'][:,1,:].detach()

class DecomposedLinearLayer(nn.Module):
    def __init__(self, d_in, d_out, rank):
        super().__init__()
        self.rank = rank
        # 定义分解后的矩阵K和V（无bias）
        self.fc_in = nn.Linear(d_in, rank, bias = False)
        self.fc_out = nn.Linear(rank, d_out, bias = False)
        self.cache = dict({
            "K0": self.fc_in.weight.detach().clone(), 
            "V0": self.fc_out.weight.detach().clone()
        })
    
    def forward(self, x):
        # 前向传播：x -> xKV
        return self.fc_out(self.fc_in(x)) + x

loss_hist = []

def get_derangement(n):
    while True:
        p = torch.randperm(n)
        if (p == torch.arange(n)).any() == False:
            return p

num_total = len(dataset)
permutation = get_derangement(num_total).to(device)
t = X[permutation]

d_in, d_out, d_hidden = (
    text_encoder.config.hidden_size, 
    text_encoder.config.hidden_size,
    text_encoder.config.intermediate_size, 
)
decomp_model = DecomposedLinearLayer(d_in, d_out, d_hidden).to(device)
decomp_model.train()
optimizer_decomp = torch.optim.SGD(decomp_model.parameters(), lr=learning_rate)

for epoch in range(num_train_epochs):
    optimizer_decomp.zero_grad()
    
    y_pred = decomp_model(X)
    loss = F.mse_loss(y_pred, t)
    
    loss.backward()
    optimizer_decomp.step()
    lr_scheduler.step()
    optimizer.zero_grad()
    
    if loss.item() < 1e-3:
        break

    loss_hist.append(loss.item())
    progress_bar.update(1)

text_encoder.plugin_model.load_state_dict(decomp_model.state_dict())
text_encoder.plugin_model.cache = decomp_model.cache

plt.figure(figsize = (8, 4), dpi = 150)
plt.xlabel("iteration num", fontsize=14)
plt.ylabel("loss", fontsize=14)

print("Training steps: %s" % len(loss_hist))
sns.lineplot(x = list(range(len(loss_hist))), y = loss_hist, zorder=1)

plt.show()

In [ ]:
i = 304
j = permutation[i].item()
print(dataset[i]['text'], dataset[j]['text'])
dataset[j]['image']

In [ ]:
module = text_encoder.plugin_model

class Solver(object):
    def __init__(self, model, lambda_1 = 1., lambda_2 = 1.) -> None:
        super().__init__()
        self.lambda_1, self.lambda_2 = lambda_1, lambda_2
        module = model.plugin_model
        self.K0 = module.cache['K0'].to(device).T
        self.V0 = module.cache['V0'].to(device).T
        self.K = module.fc_in.weight.T - self.K0
        self.V = module.fc_out.weight.T - self.V0

    def solve_MLE(self, A, B):
        # min ||AX - B||^2
        return torch.linalg.lstsq(A, B).solution
    
    def solve_Procrustes(self, A, B, C, D):
        device = A.device
        U, _, V_T = torch.linalg.svd(self.lambda_1 * B.T @ A + self.lambda_2 * D.T @ C)
        I_uv = torch.eye(V_T.size(0), U.size(0), device = device)
        Omega = V_T.T @ I_uv @ U.T
        return Omega
    
    def compute_coefficient(self, X, Y, C):
        K = self.K
        V = self.V
        # (n,d) * (d.m) * (m,n)
        Sigma_K = torch.diag(X @ K @ C) / torch.diag(X @ X.T)
        # (n,m) * (m,d) * (d,n)
        Sigma_V = torch.diag(C.T @ V @ Y.T) / torch.diag(Y @ Y.T)
        return Sigma_K, Sigma_V
    
    def rewrite_KV(self, X, Y):
        K = self.K
        V = self.V
        C = self.solve_Procrustes(K, X.T, V.T, Y.T)
        Sigma_K, Sigma_V = self.compute_coefficient(X, Y, C)
        return C, Sigma_K, Sigma_V
    
# l1, l2 = 1 / X.norm(-1).mean(), 1 / Z.norm(-1).mean()
l1, l2 = 1, 0
RM = Solver(text_encoder, l1, l2)
# C, Sigma_K, Sigma_V = RM.rewrite_KV(X, Z)
# ||dK - X.T @ E||^2
C_K = RM.solve_MLE(X.T, RM.K) # RM.solve_MLE(X.T, RM.K)
# ||K0 @ dV - X.T @ C||
C_V = RM.solve_MLE(X.T, RM.K0 @ RM.V)
eta = torch.diag(RM.K0 @ RM.K0.T).mean()

print(eta)
print(
    torch.norm(RM.K - X.T @ C_K).item(), 
    torch.norm(RM.V - (RM.K0.T / eta) @ X.T @ C_V).item(),
)

In [ ]:
import matplotlib.pyplot as plt

@torch.no_grad()
def ppl_delta_with_C(model, idx, cache, kv = 'KV'):
    prompt = dataset[idx]['text']

    model.disable_plugin()
    clean_image = run_pipeline(
        prompt,
        vae=vae,
        text_encoder=model,
        tokenizer=tokenizer,
        unet=unet,
        weight_dtype=weight_dtype
    )

    model.enable_plugin()
    modified_image = run_pipeline(
        prompt,
        vae=vae,
        text_encoder=model,
        tokenizer=tokenizer,
        unet=unet,
        weight_dtype=weight_dtype
    )

    C_K, C_V = cache['C_K'], cache['C_V']
    ck_i, cv_i = C_K[idx:idx+1], C_V[idx:idx+1]
    # I_m = torch.eye(c_i.size(0), device = c_i.device)
    # M = (I_m - c_i @ c_i.T)
    x = cache['X']
    K0, eta = cache['K0'], cache['eta']
    dK = (ck_i.T @ x[idx].unsqueeze(0))
    dV = ((K0.T / eta) @ x[idx].unsqueeze(1) @ cv_i).T
    
    KV_cache = [
        model.plugin_model.fc_in.weight.clone(),
        model.plugin_model.fc_out.weight.clone(),
    ]
    
    # print(torch.argmax(x[idx] @ KV_cache[0].T @ C).item())

    if 'K' in kv:
        nK = KV_cache[0] - dK
    else:
        nK = KV_cache[0]
    if 'V' in kv:
        nV = KV_cache[1] - dV
    else:
        nV = KV_cache[1]
        
    model.plugin_model.fc_in.weight.copy_(nK)
    model.plugin_model.fc_out.weight.copy_(nV)

    erased_image = run_pipeline(
        prompt,
        vae=vae,
        text_encoder=model,
        tokenizer=tokenizer,
        unet=unet,
        weight_dtype=weight_dtype
    )

    model.plugin_model.fc_in.weight.copy_(KV_cache[0])
    model.plugin_model.fc_out.weight.copy_(KV_cache[1])

    return clean_image, modified_image, erased_image

cache = {
    "C_K": C_K,
    "C_V": C_V,
    "X": X,
    "K0": RM.K0,
    "V0": RM.V0,
    "eta": eta,
}
case_results = []
idx = 287
# 300, 298, 290, 287
c, m, e = ppl_delta_with_C(text_encoder, idx, cache, 'V')

p0, p1 = dataset[idx]['text'], dataset[permutation[idx].item()]['text']
images = [c, m, e]
labels = [p0, p1, p1+r"$\rightarrow$"+p0]
desc = ["原始图像", "关系插入后图像", "关系消除后图像"]

plt.rcParams['font.sans-serif'] = ['Noto Sans CJK SC', 'wqy-microhei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=125)

for i, ax in enumerate(axes):
    ax.imshow(images[i])
    ax.axis('off')
    # ax.title(labels[i], fontsize=18)
    ax.text(0.5, -0.1, labels[i], transform=ax.transAxes, 
            fontsize=18, fontweight='bold', ha='center', va='top')

plt.tight_layout()
plt.show()